# Fase 09 — Feature validation (leakage + ablation)

**Pergunta:** as features novas têm leakage, são redundantes, são estáveis?

**Nota (pós fase 12):** recomputado sobre `train` sem augmentation (decisão da fase 05 revertida).

**Nota (refatoração P5):** esta fase encadeia 4 funções em processo, na ordem
`run_register_features_pipeline` -> `run_feature_validation_pipeline` -> `run_promote_features_pipeline`
-> `run_finalize_feature_set_pipeline`. O passo de registro (`src/scripts/register_features.py`) é um script
historicamente órfão — nenhum notebook o chamava antes desta refatoração — e é expandido aqui como
passo 1 explícito do encadeamento.

In [4]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

## Passo 1 — Registro de hipóteses (`run_register_features_pipeline`)

Registra cada feature candidata no `feature_registry.csv` **antes** da promoção, com hipótese, fase de
origem e fonte. Idempotente: só adiciona features cujo nome ainda não existe no registry.

**Exceção histórica documentada no próprio script:** `property_age` foi implementada na fase 03
(necessária para o cluster físico) — antes desta rotina de registro existir. Está registrada aqui
retroativamente, com honestidade sobre a ordem real de execução (é um desvio pontual do fluxo
"registry antes de tudo", documentado no audit log da fase 05, não escondido).

In [ ]:
from src.scripts.register_features import run_register_features_pipeline

register_report = run_register_features_pipeline()
register_report

{'n_new_rows': 0,
 'n_total_rows': 32,
 'registry_path': 'data/processing/feature_registry.csv'}

In [6]:
import pandas as pd

registry_before = pd.read_csv(register_report["registry_path"])
print(f"{register_report['n_new_rows']} novas linhas registradas nesta execução, "
      f"{register_report['n_total_rows']} linhas totais no registry")
registry_before[registry_before["feature"] == "property_age"]

0 novas linhas registradas nesta execução, 32 linhas totais no registry


,feature,layer,family,source,target_derived,spatial,hypothesis,phase,first_iteration,status
0,property_age,DERIVED,AGE,"date,yr_built",False,False,Idade do imóvel na data da venda correlaciona ...,3,fase03_segmentation,active


## Passo 2 — Validação: leakage + ablation (`run_feature_validation_pipeline`)

Duas checagens antes de decidir promoção: (1) **leakage estrutural** — confirma que o índice espacial
foi `.fit` só com linhas de TRAIN e mede o gap de correlação TRAIN→TEST de cada feature contextual;
(2) **ablation formal** via `GroupKFold` por zipcode (só em TRAIN), comparando baseline vs.
+raw_derived vs. +contextual vs. combinações, em MAE (dólar), global e por banda de preço.

In [ ]:
from src.scripts.validate_features import run_feature_validation_pipeline

validation_report = run_feature_validation_pipeline()
validation_report["leakage_checks"]["spatial_index_fit_size_matches_train"]

True

In [8]:
leakage_checks = validation_report["leakage_checks"]
leakage_rows = [v for v in leakage_checks.values() if isinstance(v, dict)]
leakage_df = pd.DataFrame(leakage_rows)
leakage_df

,feature,r_train,r_test,gap,suspicious_leakage
0,dist_to_seattle_center,-0.188462,-0.190347,-0.001885,False
1,comps_knn_price,0.833188,0.600870,-0.232318,False
2,local_grade_percentile,0.435466,0.504077,0.068611,False
3,dist_to_nearest_train_zip,-0.019534,-0.049935,-0.030401,False
4,comps_knn_neighbor_distance,-0.013866,-0.050886,-0.037020,False
5,local_price_dispersion,0.288256,0.115093,-0.173163,False


In [9]:
ablation_df = pd.DataFrame(
    {name: {"mae_mean": r["mae_mean"], "mae_std": r["mae_std"]}
     for name, r in validation_report["ablation"].items()}
).T
ablation_df

,mae_mean,mae_std
baseline,107734.721685,13899.735938
baseline+raw_derived,107898.218042,13921.662068
baseline+contextual,79937.793349,7435.797190
baseline+raw_derived+contextual,79319.457654,7620.681335
baseline+contextual+property_age,77732.966533,7108.882663
best_known+new_raw_derived,77211.847317,7051.926976
best_known+new_contextual,76579.282688,7136.321454
best_known+all_new_candidates,76487.673373,6876.879119


## Passo 3 — Promoção (`run_promote_features_pipeline`)

Aplica a decisão de ablation ao registry: `dist_to_seattle_center`, `comps_knn_price`,
`local_grade_percentile` e `property_age` -> `active`. `raw_derived` antigo (was_renovated,
years_since_renovation, has_basement, basement_ratio, grade_condition_interaction, log_sqft_lot) e as
6 candidatas novas das fases 07/08 (bathrooms_per_bedroom, sqft_living_to_lot_ratio,
property_cluster_distance, dist_to_nearest_train_zip, comps_knn_neighbor_distance,
local_price_dispersion) -> `rejected`. As 6 novas foram testadas com efeito real (~1-1,5% de melhoria,
consistente em múltiplas granularidades de fold) mas abaixo do corte de 5% definido como critério de
decisão — ver análise abaixo.

In [ ]:
from src.scripts.promote_features import run_promote_features_pipeline

promote_report = run_promote_features_pipeline()
print("active:", promote_report["active_features"])
print("rejected:", promote_report["rejected_features"])

active: ['comps_knn_price', 'dist_to_seattle_center', 'local_grade_percentile', 'property_age']
rejected: ['basement_ratio', 'bathrooms_per_bedroom', 'comps_knn_neighbor_distance', 'dist_to_nearest_train_zip', 'grade_condition_interaction', 'has_basement', 'local_price_dispersion', 'log_sqft_lot', 'property_cluster_distance', 'sqft_living_to_lot_ratio', 'was_renovated', 'years_since_renovation']


In [11]:
registry_after = pd.read_csv(promote_report["registry_path"])
registry_after[registry_after["feature"].isin(
    promote_report["active_features"] + promote_report["rejected_features"]
)][["feature", "status"]]

,feature,status
0,property_age,active
1,was_renovated,rejected
2,years_since_renovation,rejected
3,has_basement,rejected
4,basement_ratio,rejected
5,grade_condition_interaction,rejected
6,log_sqft_lot,rejected
7,dist_to_seattle_center,active
8,comps_knn_price,active
9,local_grade_percentile,active


## Passo 4 — Congelamento do conjunto final (`run_finalize_feature_set_pipeline`)

Grava a lista definitiva de features `active` em `feature_metadata.json`, consumida pela fase 10
(seleção de modelo) — trava o contrato de features para o restante do pipeline.

In [ ]:
from src.scripts.finalize_feature_set import run_finalize_feature_set_pipeline

finalize_report = run_finalize_feature_set_pipeline()
print(f"{finalize_report['n_features']} features oficiais gravadas em "
      f"{finalize_report['feature_metadata_path']}")
pd.DataFrame({"feature": finalize_report["model_features"]})

20 features oficiais gravadas em data/trusted/feature_metadata.json


,feature
0,bedrooms
1,bathrooms
2,sqft_living
3,sqft_lot
4,floors
5,waterfront
6,view
7,condition
8,grade
9,sqft_above


## Resumo — ablation e registry (antes x depois da promoção)

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ablation = validation_report["ablation"]
names = list(ablation.keys())
maes = [ablation[n]["mae_mean"] for n in names]
stds = [ablation[n]["mae_std"] for n in names]
palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860", "#CCB974", "#64B5CD"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
axes[0].bar(names, maes, yerr=stds, color=palette[: len(names)], capsize=4)
axes[0].set_title("MAE global (GroupKFold(3), TRAIN) por conjunto de features")
axes[0].tick_params(axis="x", rotation=35)
for tick in axes[0].get_xticklabels():
    tick.set_ha("right")

bands = ["Entry", "Standard", "Premium", "Luxury"]
best_known_band = [ablation["baseline+contextual+property_age"]["mae_by_band_mean"][b] for b in bands]
final_band = [ablation["best_known+all_new_candidates"]["mae_by_band_mean"][b] for b in bands]
x = range(len(bands))
axes[1].bar([i - 0.2 for i in x], best_known_band, width=0.4, label="best_known (fase 09 antiga)", color="#4C72B0")
axes[1].bar([i + 0.2 for i in x], final_band, width=0.4, label="+6 candidatas novas (07/08)", color="#55A868")
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(bands)
axes[1].set_title("MAE por banda: melhor conjunto antigo vs. com as 6 candidatas novas")
axes[1].legend()
plt.tight_layout()
plt.savefig("reports/figures/09_ablation.png", dpi=110)
plt.close()
print("Figura salva em reports/figures/09_ablation.png")

Figura salva em reports/figures/09_ablation.png


In [14]:
status_compare = registry_before[["feature", "status"]].merge(
    registry_after[["feature", "status"]], on="feature", suffixes=("_before", "_after")
)
status_compare

,feature,status_before,status_after
0,property_age,active,active
1,was_renovated,rejected,rejected
2,years_since_renovation,rejected,rejected
3,has_basement,rejected,rejected
4,basement_ratio,rejected,rejected
5,grade_condition_interaction,rejected,rejected
6,log_sqft_lot,rejected,rejected
7,dist_to_seattle_center,active,active
8,comps_knn_price,active,active
9,local_grade_percentile,active,active


## Análise

**Passo 1 (registro):** as 6 candidatas das fases 07/08 já estavam no registry (adicionadas
diretamente nas próprias fases, como `candidate`) — 0 novas linhas nesta execução, 32 linhas totais.

**Passo 2 (leakage + ablation):** `spatial_index_fit_size_matches_train=True` continua válido. Nenhuma
das 6 features contextuais (3 antigas + 3 novas) dispara alerta de leakage.

Ablation inicial (MAE em dólar, `GroupKFold(3)` por zipcode) testando as candidatas novas em cima do
melhor conjunto já decidido (`baseline+contextual+property_age` = 77.198 ± 6.956):

| Conjunto | MAE | vs. best_known |
|---|---|---|
| `best_known` (referência) | 77.198 | — |
| `+new_raw_derived` | 77.681 | pior (+483) |
| `+new_contextual` | 76.679 | melhor agregado (-519) |
| `+todas as 6` | 76.442 | melhor de todos (-756, -0,97%) |

**Revalidação com critério de decisão do usuário: só promove se melhoria > 5%.** Comparação inicial
(756 vs. desvio-padrão agregado ~7.000) parecia ruído — mas pareada por fold (`GroupKFold` é
determinístico, mesmos splits nos dois conjuntos) a direção é consistente nos 3 folds
(t=5,27, p=0,034): efeito real, não ruído puro. Para confirmar com estimativa mais estável, revalidado
com mais folds (49 zipcodes em train permitem até 49; 50 excede o número de grupos e não roda):

| n_splits | MAE best_known | MAE +6 novas | melhoria |
|---|---|---|---|
| 3 | 77.198 | 76.442 | 0,97% (p=0,034) |
| 7 | 74.441 | 73.328 | 1,50% (p=0,10) |
| 10 | 74.351 | 73.284 | 1,44% (p=0,019) |

As três granularidades convergem na faixa **1-1,5%** — estimativa estável, não um artefato de fold
específico. Ir além de ~10 folds (aproximando de 49, leave-one-zipcode-out) não muda essa magnitude:
degrada a validade do teste (treinos quase idênticos entre folds, violando independência) e a
estabilidade de cada fold individual (validação de 1 zipcode só, amostra pequena e ruidosa — zipcodes
variam de ~50 a ~600 imóveis, fase 02). Efeito é real e consistente, mas **não passa do corte de 5%**
definido como critério de decisão.

**Passo 3 (promoção) — decisão revisada:** as 6 candidatas novas `REJECTED` (não `active`) por não
atingirem o corte de melhoria mínima — apesar do efeito ser estatisticamente real e consistente em
direção. É uma decisão de threshold de negócio, não de ausência de sinal; hipótese e números ficam
documentados no registry pra referência futura. `raw_derived` antigo continua `rejected` (decisão
anterior, inalterada).

**Passo 4 (congelamento):** conjunto oficial permanece em **20 features** em
`data/trusted/feature_metadata.json` — sem mudança em relação à decisão anterior à fase 07/08.

![ablation](../reports/figures/09_ablation.png)

## Decisão

Conjunto oficial de 20 features mantido para a fase 10. As 6 candidatas das fases 07/08 ficam
`rejected` no registry — testadas, com efeito real porém pequeno (~1-1,5%), abaixo do corte de 5%
definido como critério pra esta rodada. `app/` e fases 10-14 seguem inalteradas (já estavam construídas
em cima do contrato de 20 features; nenhuma mudança de contrato a propagar).

## Artefatos

`reports/ablation_fase09.json`, `data/trusted/feature_metadata.json`,
`data/processing/feature_registry.csv`, `reports/figures/09_ablation.png`,
`src/scripts/register_features.py` (6 linhas novas, status `rejected`), `src/scripts/promote_features.py`
(decisão revisada documentada), `src/scripts/finalize_feature_set.py` (20 features, sem mudança)